# 🎯 Few-Shot Open-Set KWS — Training on Colab

**Full training pipeline**: Setup → Data → MFCC → DSCNN-L → Triplet Loss → Checkpoints

**GPU**: Runtime → Change runtime type → T4 GPU

## 1. Setup & Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Project directory on Drive (checkpoints persist here)
DRIVE_PROJECT = '/content/drive/MyDrive/DoAnTotNghiep'
!mkdir -p {DRIVE_PROJECT}/checkpoints {DRIVE_PROJECT}/results

In [ ]:
# Upload your project as a zip, or clone from Git
# Option A: Upload zip (run this, then upload DoAnTotNghiep.zip)
# from google.colab import files
# uploaded = files.upload()
# !unzip -q DoAnTotNghiep.zip -d /content/

# Option B: Clone from GitHub (uncomment & fill your repo URL)
# !git clone https://github.com/YOUR_USER/DoAnTotNghiep.git /content/DoAnTotNghiep

# Option C: Copy from Drive
import shutil, os
if os.path.exists(f'{DRIVE_PROJECT}/DoAnTotNghiep.zip'):
    !unzip -qo {DRIVE_PROJECT}/DoAnTotNghiep.zip -d /content/
    print('Extracted from Drive')
else:
    print('Please upload your project. Use one of the options above.')
    print(f'Or place DoAnTotNghiep.zip in {DRIVE_PROJECT}/')

In [ ]:
%cd /content/DoAnTotNghiep
!pip install -q torch torchaudio numpy pyyaml matplotlib tensorboard scikit-learn soundfile requests tqdm
# speechbrain & noisereduce are optional for training

import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}, Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')

## 2. Download Datasets

In [ ]:
%%time
# Download Google Speech Commands v2 (~2.3GB)
!python data/download_gsc.py

In [ ]:
%%time
# Download DEMAND noise dataset
import os, zipfile, requests
from pathlib import Path
from tqdm import tqdm

DEMAND_DIR = Path('data/demand')
DEMAND_DIR.mkdir(parents=True, exist_ok=True)

if not any(DEMAND_DIR.glob('**/*.wav')):
    DEMAND_URL = 'https://zenodo.org/records/1227121/files/DEMAND.zip'
    zip_path = Path('data/DEMAND.zip')

    if not zip_path.exists():
        print('Downloading DEMAND dataset...')
        r = requests.get(DEMAND_URL, stream=True, timeout=120)
        r.raise_for_status()
        total = int(r.headers.get('content-length', 0))
        with open(zip_path, 'wb') as f, tqdm(total=total, unit='B', unit_scale=True) as pbar:
            for chunk in r.iter_content(8192):
                f.write(chunk)
                pbar.update(len(chunk))

    print('Extracting DEMAND...')
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall('data/')

    # Move WAV files to flat directory
    for wav in Path('data').rglob('*.wav'):
        if 'demand' not in str(wav).lower() and 'DEMAND' in str(wav):
            dest = DEMAND_DIR / wav.name
            if not dest.exists():
                wav.rename(dest)

noise_count = len(list(DEMAND_DIR.rglob('*.wav')))
print(f'DEMAND noise files: {noise_count}')

In [ ]:
%%time
# Download MSWC English
# MSWC is large -- we download only top words via the MLCommons dataset
# Using torchaudio's built-in SPEECHCOMMANDS for now as a working alternative

import torchaudio

MSWC_DIR = Path('data/mswc_en')
MSWC_DIR.mkdir(parents=True, exist_ok=True)

# Note: MSWC download requires manual setup or MLCommons API.
# For immediate training, we use GSC v2 as a proxy dataset.
# Replace this section when MSWC access is configured.
print('MSWC requires manual download from MLCommons.')
print('For now, training will use GSC v2 words as training data.')
print('This is sufficient for development & debugging the pipeline.')

## 3. Build Dataset & DataLoader

In [ ]:
import sys
sys.path.insert(0, '/content/DoAnTotNghiep')

import random
import numpy as np
import torch
import torchaudio
import yaml
from pathlib import Path
from torch.utils.data import Dataset, DataLoader

from src.features.mfcc import MFCCExtractor
from src.features.augmentation import NoiseAugmenter
from src.models.dscnn import DSCNN
from src.models.prototypical import EpisodicBatchSampler, TripletLoss, train_one_epoch

# Load config
with open('configs/default.yaml') as f:
    cfg = yaml.safe_load(f)

print('Config loaded:', list(cfg.keys()))

In [ ]:
class KWSAudioDataset(Dataset):
    """Audio dataset for episodic training.

    Loads WAV files, pads/trims to 1 sec, extracts MFCC features.
    Optionally applies noise augmentation during training.
    """

    def __init__(
        self,
        data_dir: Path,
        words: list[str],
        mfcc_extractor: MFCCExtractor,
        augmenter: NoiseAugmenter | None = None,
        max_samples_per_word: int = 200,
    ):
        self.mfcc = mfcc_extractor
        self.augmenter = augmenter
        self.files: list[Path] = []
        self.labels: list[int] = []
        self.word_to_idx: dict[str, int] = {}

        for i, word in enumerate(sorted(words)):
            self.word_to_idx[word] = i
            word_dir = data_dir / word
            if not word_dir.exists():
                print(f'  Warning: {word_dir} not found, skipping')
                continue
            wavs = sorted(word_dir.glob('*.wav'))[:max_samples_per_word]
            for wav in wavs:
                self.files.append(wav)
                self.labels.append(i)

        print(f'Dataset: {len(self.files)} samples, {len(set(self.labels))} classes')

    def __len__(self) -> int:
        return len(self.files)

    def __getitem__(self, idx: int) -> tuple[torch.Tensor, int]:
        waveform, sr = torchaudio.load(self.files[idx])

        # Resample if needed
        if sr != 16000:
            waveform = torchaudio.transforms.Resample(sr, 16000)(waveform)

        # Mono
        if waveform.shape[0] > 1:
            waveform = waveform.mean(dim=0, keepdim=True)

        # Noise augmentation (training only)
        if self.augmenter is not None:
            waveform = self.augmenter.augment(waveform)

        # Extract MFCC: (1, T) -> (1, 47, 10)
        mfcc = self.mfcc.extract(waveform)  # (1, 47, 10)
        return mfcc, self.labels[idx]

In [ ]:
# Initialize feature extractor & augmenter
mfcc_extractor = MFCCExtractor(
    n_mfcc=cfg['mfcc']['n_mfcc'],
    num_features=cfg['mfcc']['num_features'],
)

# Noise augmenter (if DEMAND downloaded)
demand_dir = Path(cfg['noise']['demand_dir'])
if demand_dir.exists() and any(demand_dir.rglob('*.wav')):
    augmenter = NoiseAugmenter(
        noise_dir=demand_dir,
        prob=cfg['noise']['prob'],
        snr_db=cfg['noise']['snr_db'],
    )
    print(f'Noise augmenter: {len(augmenter.noise_files)} files, p={augmenter.prob}, SNR={augmenter.snr_db}dB')
else:
    augmenter = None
    print('No DEMAND noise found — training without augmentation')

# Verify MFCC output
test_wav = torch.randn(1, 16000)
test_mfcc = mfcc_extractor.extract(test_wav)
print(f'MFCC test: input {test_wav.shape} -> output {test_mfcc.shape}')
assert test_mfcc.shape == (1, 47, 10), f'Expected (1, 47, 10), got {test_mfcc.shape}'

In [ ]:
# Build dataset using GSC v2 words (or MSWC when available)
GSC_DIR = Path(cfg['data']['gsc_dir'])

# Use GSC words for training (30 words, excluding 5 reserved)
EXCLUDED = {'backward', 'forward', 'visual', 'follow', 'learn'}
all_words = sorted([d.name for d in GSC_DIR.iterdir()
                    if d.is_dir() and not d.name.startswith('_')])
train_words = [w for w in all_words if w not in EXCLUDED]
print(f'Training words ({len(train_words)}): {train_words}')

# Create dataset
train_dataset = KWSAudioDataset(
    data_dir=GSC_DIR,
    words=train_words,
    mfcc_extractor=mfcc_extractor,
    augmenter=augmenter,
    max_samples_per_word=200,  # limit for faster training
)

# Episodic batch sampler — adjust n_classes if fewer words available
n_classes_available = len(set(train_dataset.labels))
n_classes_per_episode = min(cfg['training']['n_classes'], n_classes_available)
n_samples_per_class = cfg['training']['n_samples']

print(f'Episodic sampling: {n_classes_per_episode} classes x {n_samples_per_class} samples')

sampler = EpisodicBatchSampler(
    labels=train_dataset.labels,
    n_classes=n_classes_per_episode,
    n_samples=n_samples_per_class,
    n_episodes=cfg['training']['episodes_per_epoch'],
)

train_loader = DataLoader(
    train_dataset,
    batch_sampler=sampler,
    num_workers=2,
    pin_memory=True,
)

## 4. Model & Training Setup

In [ ]:
# Seed for reproducibility
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

set_seed(cfg['seed'])

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

# Model
model_size = cfg['model']['architecture'][-1]  # 'L'
encoder = DSCNN(model_size=model_size).to(device)
param_count = sum(p.numel() for p in encoder.parameters())
print(f'DSCNN-{model_size}: {param_count:,} parameters')

# Verify model output
with torch.no_grad():
    test_input = torch.randn(2, 1, 47, 10).to(device)
    test_output = encoder(test_input)
    print(f'Model test: {test_input.shape} -> {test_output.shape}')
    assert test_output.shape == (2, 276), f'Expected (2, 276), got {test_output.shape}'

In [ ]:
# Optimizer, Scheduler, Loss
train_cfg = cfg['training']
optimizer = torch.optim.Adam(encoder.parameters(), lr=train_cfg['optimizer']['lr'])
scheduler = torch.optim.lr_scheduler.StepLR(
    optimizer,
    step_size=train_cfg['scheduler']['step_size'],
    gamma=train_cfg['scheduler']['gamma'],
)
loss_fn = TripletLoss(margin=train_cfg['triplet_margin'])

print(f"Optimizer: Adam lr={train_cfg['optimizer']['lr']}")
print(f"Scheduler: StepLR(step={train_cfg['scheduler']['step_size']}, gamma={train_cfg['scheduler']['gamma']})")
print(f"Loss: TripletLoss(margin={train_cfg['triplet_margin']})")
print(f"Plan: {train_cfg['epochs']} epochs x {train_cfg['episodes_per_epoch']} episodes")

## 5. Training Loop

In [ ]:
import time
import torch.nn.functional as F
from torch.utils.tensorboard import SummaryWriter

CHECKPOINT_DIR = Path(DRIVE_PROJECT) / 'checkpoints'
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

LOG_DIR = Path('runs') / 'kws_training'
writer = SummaryWriter(log_dir=str(LOG_DIR))

NUM_EPOCHS = train_cfg['epochs']  # 40
SAVE_EVERY = cfg['checkpoint']['save_every']  # 5
best_loss = float('inf')

print(f'Training for {NUM_EPOCHS} epochs')
print(f'Checkpoints: {CHECKPOINT_DIR}')
print(f'TensorBoard: {LOG_DIR}')
print('=' * 60)

for epoch in range(NUM_EPOCHS):
    t0 = time.time()

    # Train one epoch
    metrics = train_one_epoch(encoder, train_loader, optimizer, loss_fn, device)
    scheduler.step()

    elapsed = time.time() - t0
    current_lr = optimizer.param_groups[0]['lr']

    # Log
    writer.add_scalar('Loss/train', metrics['loss'], epoch)
    writer.add_scalar('LR', current_lr, epoch)

    print(
        f"Epoch {epoch+1:2d}/{NUM_EPOCHS} | "
        f"Loss: {metrics['loss']:.4f} | "
        f"LR: {current_lr:.6f} | "
        f"Episodes: {metrics['num_episodes']} | "
        f"Time: {elapsed:.1f}s"
    )

    # Save checkpoint
    if (epoch + 1) % SAVE_EVERY == 0 or (epoch + 1) == NUM_EPOCHS:
        ckpt = {
            'epoch': epoch,
            'model_state_dict': encoder.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'loss': metrics['loss'],
            'val_auc': 0.0,  # placeholder until validation is wired
        }

        # Save latest
        torch.save(ckpt, CHECKPOINT_DIR / 'latest.pt')

        # Save best by loss (until val_auc is available)
        if metrics['loss'] < best_loss:
            best_loss = metrics['loss']
            torch.save(ckpt, CHECKPOINT_DIR / 'best.pt')
            print(f'  ★ New best loss: {best_loss:.4f}')

        # Save epoch checkpoint
        torch.save(ckpt, CHECKPOINT_DIR / f'epoch_{epoch+1:02d}.pt')
        print(f'  Checkpoint saved to {CHECKPOINT_DIR}')

writer.close()
print('\n' + '=' * 60)
print(f'Training complete! Best loss: {best_loss:.4f}')
print(f'Checkpoints saved to: {CHECKPOINT_DIR}')

## 6. Quick Verification

In [ ]:
# Load best checkpoint and verify embedding quality
best_ckpt = torch.load(CHECKPOINT_DIR / 'best.pt', map_location=device, weights_only=False)
encoder.load_state_dict(best_ckpt['model_state_dict'])
encoder.eval()
print(f"Loaded best checkpoint (epoch {best_ckpt['epoch']+1}, loss {best_ckpt['loss']:.4f})")

# Test: generate embeddings for a few samples
with torch.no_grad():
    sample_mfcc = torch.randn(5, 1, 47, 10).to(device)
    embeddings = encoder(sample_mfcc)
    embeddings = F.normalize(embeddings, p=2, dim=-1)

    print(f'Embedding shape: {embeddings.shape}')  # (5, 276)
    print(f'L2 norms: {embeddings.norm(dim=-1).cpu().numpy()}')
    print(f'Pairwise distances (sample):')
    dists = torch.cdist(embeddings, embeddings)
    print(dists.cpu().numpy().round(3))

In [ ]:
# Test with real audio from GSC
GSC_DIR = Path('data/gsc_v2')

test_words = ['yes', 'no', 'stop']
word_embeddings = {}

for word in test_words:
    word_dir = GSC_DIR / word
    if not word_dir.exists():
        continue
    wavs = sorted(word_dir.glob('*.wav'))[:5]  # 5-shot
    embs = []
    for wav in wavs:
        waveform, sr = torchaudio.load(wav)
        if sr != 16000:
            waveform = torchaudio.transforms.Resample(sr, 16000)(waveform)
        mfcc = mfcc_extractor.extract(waveform)
        mfcc = mfcc.unsqueeze(0).to(device)  # (1, 1, 47, 10)
        with torch.no_grad():
            emb = encoder(mfcc)
            emb = F.normalize(emb, p=2, dim=-1)
        embs.append(emb.squeeze(0))

    prototype = torch.stack(embs).mean(dim=0)
    word_embeddings[word] = prototype
    print(f'{word}: prototype from {len(embs)} samples, norm={prototype.norm():.4f}')

# Compute inter-prototype distances
words = list(word_embeddings.keys())
for i, w1 in enumerate(words):
    for w2 in words[i+1:]:
        d = F.pairwise_distance(
            word_embeddings[w1].unsqueeze(0),
            word_embeddings[w2].unsqueeze(0)
        ).item()
        print(f'  d({w1}, {w2}) = {d:.4f}')

## 7. TensorBoard

In [ ]:
%load_ext tensorboard
%tensorboard --logdir runs/

## 8. Copy Results to Drive

In [ ]:
# Copy TensorBoard logs and any results to Drive
import shutil

drive_logs = Path(DRIVE_PROJECT) / 'runs'
if LOG_DIR.exists():
    if drive_logs.exists():
        shutil.rmtree(drive_logs)
    shutil.copytree(LOG_DIR, drive_logs)
    print(f'TensorBoard logs copied to {drive_logs}')

print(f'\nCheckpoints on Drive: {CHECKPOINT_DIR}')
for f in sorted(CHECKPOINT_DIR.glob('*.pt')):
    size_mb = f.stat().st_size / 1024 / 1024
    print(f'  {f.name}: {size_mb:.1f}MB')